# Agent Discovery Walkthrough - Vantara Commerce

This notebook provides an interactive walkthrough of AI agent discovery using Briefcase AI. We'll demonstrate how to automatically discover and catalog AI agents across a large e-commerce organization without requiring IT access.

## Problem Context

Vantara Commerce has 45 product teams independently deploying AI features. The governance team needs to answer:
- How many AI models are running in production?
- Which vendors power our AI applications?
- Who owns each AI system?
- Are there any "shadow AI" deployments?

Traditional discovery methods require:
- Cloud account access (often restricted)
- Manual surveying of teams (incomplete and outdated)
- Infrastructure scanning (complex and limited)

## The Briefcase AI Solution

Briefcase AI creates a self-populating agent registry through decision capture. Every AI agent registers itself automatically on first execution, building an always-current inventory without IT access.

## Setup and Initialization

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import random

# Add shared module to path
sys.path.append(os.path.join('..', 'shared'))

# Import our demo modules
import backend
from backend import briefcase_ai, COMPANY, TEAMS

# Set deterministic random seed
random.seed(42)

print(f"Briefcase AI Agent Discovery Demo")
print(f"Company: {COMPANY['name']}")
print(f"Industry: {COMPANY['industry']}")
print(f"Teams: {COMPANY['team_count']}")
print(f"Estimated monthly AI decisions: {COMPANY['monthly_ai_decisions_estimate']:,}")

In [ ]:
# Initialize Briefcase AI SDK
try:
    briefcase_ai.init()
    print("SUCCESS: Briefcase AI SDK initialized")
except Exception as e:
    print(f"INFO: Using mock implementation ({e})")

# Get backend for storage
backend_instance = backend.get_backend()
print("SUCCESS: In-memory SQLite backend configured")

## Agent Configuration Data

Let's examine the AI agents we'll discover across Vantara's teams. In a real scenario, this data would be captured automatically as agents execute their first decisions.

In [ ]:
# Load the agent configuration (normally this would be discovered automatically)
from example import AGENTS_CONFIG

# Convert to pandas DataFrame for analysis
agents_df = pd.DataFrame(AGENTS_CONFIG)

print(f"Total agents to discover: {len(agents_df)}")
print(f"\nAgent distribution by team:")
print(agents_df['team'].value_counts())

print(f"\nAgent distribution by vendor:")
print(agents_df['vendor'].value_counts())

print(f"\nAgent distribution by environment:")
print(agents_df['environment'].value_counts())

In [ ]:
# Visualize agent distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Vantara Commerce AI Agent Distribution', fontsize=16)

# By vendor
agents_df['vendor'].value_counts().plot(kind='bar', ax=axes[0,0], color='skyblue')
axes[0,0].set_title('Agents by AI Vendor')
axes[0,0].set_ylabel('Number of Agents')
axes[0,0].tick_params(axis='x', rotation=45)

# By environment
agents_df['environment'].value_counts().plot(kind='pie', ax=axes[0,1], autopct='%1.1f%%')
axes[0,1].set_title('Agents by Environment')

# By deployment type
agents_df['deployment_type'].value_counts().plot(kind='bar', ax=axes[1,0], color='lightgreen')
axes[1,0].set_title('Agents by Discovery Method')
axes[1,0].set_ylabel('Number of Agents')
axes[1,0].tick_params(axis='x', rotation=45)

# Daily decisions distribution
agents_df['estimated_daily_decisions'].plot(kind='hist', ax=axes[1,1], bins=10, color='orange', alpha=0.7)
axes[1,1].set_title('Daily Decision Volume Distribution')
axes[1,1].set_xlabel('Estimated Daily Decisions')
axes[1,1].set_ylabel('Number of Agents')

plt.tight_layout()
plt.show()

## Shadow AI Analysis

One of the key benefits of automated discovery is identifying "shadow AI" - agents deployed without governance knowledge.

In [ ]:
# Identify shadow AI agents
shadow_agents = agents_df[agents_df['was_previously_known'] == False]
known_agents = agents_df[agents_df['was_previously_known'] == True]

print(f"Known agents: {len(known_agents)}")
print(f"Shadow AI agents: {len(shadow_agents)}")
print(f"Shadow AI percentage: {len(shadow_agents)/len(agents_df)*100:.1f}%")

print(f"\nShadow AI agents details:")
for _, agent in shadow_agents.iterrows():
    print(f"  - {agent['agent_name']} ({agent['team']}) - {agent['environment']} environment")
    print(f"    Vendor: {agent['vendor']}, Model: {agent['model']}")
    print(f"    Daily decisions: {agent['estimated_daily_decisions']:,}")
    print()

## Running the Discovery Process

Now let's run the actual agent discovery simulation and store the results.

In [ ]:
# Import and run the discovery simulation
from example import simulate_agent_discovery

print("Running agent discovery simulation...")
discovered_agents = simulate_agent_discovery()

print(f"SUCCESS: Discovered {len(discovered_agents)} agents")
print(f"\nStoring agents in audit trail...")

# Store all discoveries in the backend
stored_decision_ids = []
for agent in discovered_agents:
    decision_id = backend_instance.store_decision(agent)
    stored_decision_ids.append(decision_id)

print(f"SUCCESS: {len(stored_decision_ids)} agent records stored in audit trail")

## Analysis of Discovery Results

Let's analyze the discovered agents and extract insights for governance.

In [ ]:
# Extract data from discovered agents for analysis
discovery_data = []
for agent in discovered_agents:
    agent_data = {
        'agent_name': agent.inputs[1].value,
        'team': agent.inputs[2].value,
        'vendor': agent.inputs[3].value,
        'model': agent.inputs[4].value,
        'environment': agent.inputs[5].value,
        'registered_by': agent.inputs[6].value,
        'first_seen': agent.inputs[7].value,
        'deployment_type': agent.inputs[8].value,
        'daily_decisions': int(agent.inputs[9].value),
        'was_previously_known': agent.outputs[1].value == 'True',
        'governance_record_id': agent.outputs[2].value,
        'decision_id': agent.decision_id
    }
    discovery_data.append(agent_data)

discovery_df = pd.DataFrame(discovery_data)

# Display summary statistics
print("DISCOVERY SUMMARY:")
print(f"Total agents discovered: {len(discovery_df)}")
print(f"Teams with AI: {discovery_df['team'].nunique()}")
print(f"AI vendors in use: {discovery_df['vendor'].nunique()}")
print(f"Total daily decisions: {discovery_df['daily_decisions'].sum():,}")
print(f"Shadow AI agents: {(~discovery_df['was_previously_known']).sum()}")

print(f"\nTop decision volume agents:")
top_agents = discovery_df.nlargest(5, 'daily_decisions')[['agent_name', 'team', 'daily_decisions']]
for _, agent in top_agents.iterrows():
    print(f"  {agent['agent_name']:<25} ({agent['team']:<20}) {agent['daily_decisions']:>8,} decisions/day")

## Risk Analysis

Identify potential risks from the discovered agents.

In [ ]:
# Risk analysis
print("RISK ANALYSIS:")
print("=" * 50)

# Shadow AI risks
shadow_agents = discovery_df[~discovery_df['was_previously_known']]
print(f"\n1. SHADOW AI RISK ({len(shadow_agents)} agents):")
for _, agent in shadow_agents.iterrows():
    print(f"   [!] {agent['agent_name']} - {agent['environment']} environment")
    print(f"       Team: {agent['team']}, Model: {agent['vendor']}/{agent['model']}")
    print(f"       Risk: Untracked AI with {agent['daily_decisions']:,} daily decisions")

# Experimental agents in production-adjacent environments
exp_agents = discovery_df[discovery_df['environment'] == 'experiment']
print(f"\n2. EXPERIMENTAL AGENTS RISK ({len(exp_agents)} agents):")
for _, agent in exp_agents.iterrows():
    print(f"   [!] {agent['agent_name']} using {agent['model']}")
    print(f"       Risk: Premium model usage without cost controls")

# Staging agents with production exposure
staging_agents = discovery_df[discovery_df['environment'] == 'staging']
staging_stream = staging_agents[staging_agents['deployment_type'] == 'discovered_via_output_stream']
print(f"\n3. STAGING-PRODUCTION LEAKAGE ({len(staging_stream)} agents):")
for _, agent in staging_stream.iterrows():
    print(f"   [!] {agent['agent_name']} detected in production streams")
    print(f"       Risk: Staging agent potentially processing production data")

# Vendor concentration risk
vendor_counts = discovery_df['vendor'].value_counts()
max_vendor_pct = (vendor_counts.max() / len(discovery_df)) * 100
print(f"\n4. VENDOR CONCENTRATION RISK:")
print(f"   Largest vendor: {vendor_counts.index[0]} ({vendor_counts.iloc[0]} agents, {max_vendor_pct:.1f}%)")
if max_vendor_pct > 50:
    print(f"   [!] High concentration risk - consider diversification")
else:
    print(f"   [OK] Reasonable vendor diversification")

## Audit Trail Verification

Verify that all discovered agents are properly stored and retrievable from the audit trail.

In [ ]:
# Verify audit trail integrity
print("AUDIT TRAIL VERIFICATION:")
print("=" * 50)

verification_results = []
for decision_id in stored_decision_ids:
    retrieved_agent = backend_instance.load_decision(decision_id)
    if retrieved_agent:
        agent_name = retrieved_agent.inputs[1].value
        team_name = retrieved_agent.inputs[2].value
        verification_results.append({
            'decision_id': decision_id,
            'agent_name': agent_name,
            'team': team_name,
            'retrieved': True
        })
    else:
        verification_results.append({
            'decision_id': decision_id,
            'agent_name': 'UNKNOWN',
            'team': 'UNKNOWN',
            'retrieved': False
        })

verification_df = pd.DataFrame(verification_results)
success_rate = verification_df['retrieved'].mean() * 100

print(f"Verification Results:")
print(f"  Total records tested: {len(verification_df)}")
print(f"  Successfully retrieved: {verification_df['retrieved'].sum()}")
print(f"  Success rate: {success_rate:.1f}%")

if success_rate == 100:
    print(f"  [SUCCESS] All agent records retrievable from audit trail")
else:
    print(f"  [WARNING] Some records could not be retrieved")
    failed_records = verification_df[~verification_df['retrieved']]
    print(f"  Failed retrievals: {len(failed_records)}")

## Generate Full Discovery Report

Finally, let's generate the complete discovery report as it would appear to governance teams.

In [ ]:
# Generate the full discovery report
from example import print_discovery_report

print("\n" + "=" * 80)
print("FULL GOVERNANCE REPORT")
print("=" * 80)

print_discovery_report(discovered_agents, backend_instance)

## Key Takeaways

This walkthrough demonstrated several key capabilities of Briefcase AI for agent discovery:

### 1. Zero IT Access Discovery
- No cloud account credentials required
- No infrastructure scanning needed
- Self-populating registry through normal AI operations

### 2. Shadow AI Detection
- Automatically identified 2 previously unknown agents
- Found experimental agents using premium models without cost controls
- Detected staging agents potentially processing production data

### 3. Risk Assessment
- Vendor concentration analysis
- Environment isolation validation
- Cost exposure identification
- Compliance gap detection

### 4. Audit Trail Integrity
- 100% retrieval success rate for all discovered agents
- Immutable records with governance metadata
- Complete lineage from discovery to storage

### 5. Operational Benefits
- **12.6M daily decisions** now tracked across 10 agents
- **Real-time discovery** replaces manual quarterly surveys
- **Automatic compliance** documentation for regulatory requirements
- **Immediate risk flagging** for governance review

## Next Steps

- **[Cost Attribution](../02_cost_attribution/)**: Analyze spending by discovered agents
- **[Drift Detection](../03_peak_season_drift/)**: Monitor performance of registered agents
- **[Governance Reporting](../04_governance_report/)**: Generate compliance reports including agent inventory

This discovery capability provides the foundation for comprehensive AI governance across the enterprise.